In [1]:
import matplotlib.pyplot as plt
import numpy as np
import copy
import EXOSIMS.MissionSim
import astropy.units as u
import scipy
from matplotlib import ticker
import matplotlib.colors
import seaborn as sns
import os
import warnings
plt.rcParams.update({"figure.max_open_warning": 0})

lam=630*u.nm
D=6.5*u.m
core_area=np.pi*(lam/D/2)**2*u.radian**2
IWA=(2.8*lam/D*u.radian).to(u.arcsecond)
OWA=(11*lam/D*u.radian)

In [4]:
specs7 = {
    "skipSaturationCalcs": False,
    "filterBinaries": False,
    "ppFact": 0.01,
    "eta": 3,
    #"arange": [0.09084645, 30],
    "smaknee": 10,
    "FoV": 3.0,
    "koAngles_Sun": [40, 120], # set solar keepout angles
    "missionLife": 3,  # set mission duration to 1 year
    "fillPhotometry":True,
    "missionPortion": 1,  # allocate 100% of available time to exoplanet imaging
      "pupilDiam":D.to(u.m).value,
      "starlightSuppressionSystems": [
    { "name": "imager",
      "lam": lam.to(u.nm).value,
      "BW": 0.05,
      "occ_trans": 0.2,
      "core_thruput": 0.1,
      "core_contrast": 4e-9,
      "core_area": core_area.to(u.arcsecond**2).value,
      "IWA": IWA.to(u.arcsecond).value,
      "OWA": OWA.to(u.arcsecond).value,
    }
  ],
    "modules": {
        "PlanetPopulation": "SAG13",
        "StarCatalog": "EXOCAT1",
        "OpticalSystem": " ",
        "ZodiacalLight": " ",
        "BackgroundSources": " ",
        "PlanetPhysicalModel": " ",
        "Observatory": "ObservatoryTESScsv",
        "TimeKeeping": " ",
        "PostProcessing": " ",
        "Completeness": "BrownCompleteness",
        "TargetList": " ",
        "SimulatedUniverse": "SAG13Universe",
        "SurveySimulation": " ",
        "SurveyEnsemble": " ",
    }
}
sim7 = EXOSIMS.MissionSim.MissionSim(**copy.deepcopy(specs7))
sim7.run_sim()
sim=sim7

Imported SurveySimulation (prototype) from EXOSIMS.Prototypes.SurveySimulation
Numpy random seed is: 825521459
Imported SAG13Universe (implementation) from EXOSIMS.SimulatedUniverse.SAG13Universe
Imported TargetList (prototype) from EXOSIMS.Prototypes.TargetList
Imported EXOCAT1 (implementation) from EXOSIMS.StarCatalog.EXOCAT1
Imported OpticalSystem (prototype) from EXOSIMS.Prototypes.OpticalSystem
Imported ZodiacalLight (prototype) from EXOSIMS.Prototypes.ZodiacalLight
Imported PostProcessing (prototype) from EXOSIMS.Prototypes.PostProcessing
Imported BackgroundSources (prototype) from EXOSIMS.Prototypes.BackgroundSources
Imported BrownCompleteness (implementation) from EXOSIMS.Completeness.BrownCompleteness
Imported SAG13 (implementation) from EXOSIMS.PlanetPopulation.SAG13
Imported PlanetPhysicalModel (prototype) from EXOSIMS.Prototypes.PlanetPhysicalModel
Loading cached completeness file from "/Users/Yael/.EXOSIMS/cache/v3.7.0alpha/SAG13PlanetPhysicalModelBrownCompleteness10000000

/Users/Yael/Desktop/EXOSIMS/EXOSIMS/MissionSim.py:275: UserWarning: 
The following init keywords were not found in any outspec:
	       haloStartTime (ObservatoryTESScsv)

  warnings.warn(warnstr)


  Observation #3, star ind 213 (of 523) with 4 planet(s), mission time at Obs start: 139.14 d, exoplanetObsTime: 139.14 d
   - Detected planet inds [395] (1/4)
   - Charact. planet inds [395] (1/1 detected)
  Observation #4, star ind 255 (of 523) with 0 planet(s), mission time at Obs start: 193.15 d, exoplanetObsTime: 193.15 d
  Observation #5, star ind 148 (of 523) with 0 planet(s), mission time at Obs start: 245.15 d, exoplanetObsTime: 245.15 d
  Observation #6, star ind 387 (of 523) with 3 planet(s), mission time at Obs start: 297.15 d, exoplanetObsTime: 297.15 d
   - Detected planet inds [753] (1/3)
  Observation #7, star ind 449 (of 523) with 4 planet(s), mission time at Obs start: 349.15 d, exoplanetObsTime: 349.15 d
   - Detected planet inds [882 884] (2/4)
   - Charact. planet inds [882 884] (2/2 detected)
  Observation #8, star ind 321 (of 523) with 4 planet(s), mission time at Obs start: 403.18 d, exoplanetObsTime: 403.18 d
  Observation #9, star ind 443 (of 523) with 2 plane

In [ ]:
def run_ensemble(scriptfile, N=50):
    yields = []
    sims =[]
    for i in range(N):
        print(f"Running simulation {i+1}/{N}...")
        try:
            y,sim = run_simulation(scriptfile)
            sims.append(sim)
            if y >= 0:
                yields.append(y)
            else:
                print(f"Simulation {i+1} returned invalid yield: {y}")
        except Exception as e:
            print(f"Simulation {i+1} failed: {e}")
    return yields, sims

def plot_yield_histogram(yields):
    if not yields:
        print("No yield data to plot.")
        return
    bins = range(min(yields), max(yields) + 2)
    plt.figure(figsize=(8, 5))
    plt.hist(yields, bins=bins, edgecolor='black', align='left')
    plt.xlabel("Number of Detections")
    plt.ylabel("Number of Simulations")
    plt.title("Histogram of Exoplanet Detection Yields")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

if __name__ == '__main__':
    scriptfile = os.path.join(EXOSIMS.__path__[0], 'Scripts', 'ultramarine.json')
    yields, sims = run_ensemble(scriptfile, N=10)
    print("Yields:", yields)
    plot_yield_histogram(yields)